In [ ]:
!pip install torchinfo

import math
import copy
import torch
import inspect
import pandas as pd
import torch.nn as nn

from tqdm import tqdm
from torchinfo import summary
from google.colab import files
from datasets import load_dataset
from transformers import BertModel, BertConfig
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import AutoTokenizer, DataCollatorForSeq2Seq


In [ ]:
# 허깅페이스
# 사전학습된 토크나이저와 모델을 사용할 수 있다.
bert = BertModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
input = tokenizer('good', return_tensors = 'pt')
bert(**input).pooler_output

tensor([[-7.9861e-01, -1.8929e-01,  2.1596e-01,  5.7047e-01, -8.8907e-02,
         -8.1016e-02,  8.6770e-01,  2.1602e-01,  6.7556e-02, -9.9950e-01,
          1.1877e-02,  2.6642e-01,  9.4208e-01, -2.2614e-01,  8.6260e-01,
         -4.8711e-01, -1.1770e-01, -4.5599e-01,  3.2690e-01, -6.9485e-01,
          4.3719e-01,  9.0341e-01,  4.7835e-01,  1.9232e-01,  3.0345e-01,
          3.7363e-01, -5.4194e-01,  8.5729e-01,  9.1606e-01,  6.2662e-01,
         -5.7939e-01,  1.8667e-01, -9.6357e-01, -1.7327e-01,  3.8748e-01,
         -9.5151e-01,  8.1870e-02, -6.9352e-01,  3.3270e-02,  1.1713e-02,
         -8.1613e-01,  1.8226e-01,  9.9452e-01, -3.2117e-01, -2.0109e-02,
         -3.0915e-01, -9.9958e-01,  2.2747e-01, -7.9810e-01, -1.6276e-01,
         -1.2220e-02, -3.6487e-01,  1.3833e-01,  3.5889e-01,  2.9250e-01,
          3.3502e-01, -8.6409e-02,  1.5714e-01, -2.6227e-02, -4.8066e-01,
         -5.0231e-01,  1.2959e-01, -4.3057e-02, -8.2841e-01, -3.7551e-02,
         -4.8694e-01, -7.7021e-02, -1.

In [ ]:
# 심화: 허깅페이스의 모델과 나의 모델을 섞어서 선언할 수 있다.
import torch.nn.functional as F

class CustomModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.bert = BertModel.from_pretrained('bert-base-uncased')
    self.classifier = nn.Linear(bert.config.hidden_size, 2)
    self.softmax = nn.Softmax(dim=-1)

  def forward(self, x):
    x = self.bert(**x)
    x = self.classifier((x.pooler_output))
    x = self.softmax(x)
    return x

input = tokenizer('good', return_tensors = 'pt')
mymodel = CustomModel()
mymodel(input)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([[0.6012, 0.3988]], grad_fn=<SoftmaxBackward0>)

In [ ]:
# 원하는 파라미터를 수정할 수 있다.
config = BertConfig.from_pretrained('bert-base-uncased',
                                    num_hidden_layers=6,      # 12 -> 6개로 축소
                                    hidden_dropout_prob=0.2,   # Dropout 0.1 -> 0.2로 변경
                                    output_hidden_states=True
                                    )
# 파라미터를 수정하여 모델과 토크나이저에 적용할 수 있다.
bert = BertModel.from_pretrained("bert-base-uncased", config = config)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", config = config)
# 이때, 모델의 가중치가 바뀌도록 설정됐다면 사전학습을 다시 해야한다.


# 아예 새로운 모델을 선언할 수도 있다.
bert2 = BertModel(config)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                                                       | Status     |  | 
--------------------------------------------------------------------------+------------+--+-
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.attention.self.key.bias           | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.intermediate.dense.bias           | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.attention.output.LayerNorm.weight | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.output.LayerNorm.bias             | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.attention.output.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias                                      | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias                                  | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.attention.self.value.bias         | UNEXPECTED |  | 
bert.enco

In [ ]:
bert2

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.2, inplace=False)
    

In [ ]:
# config : 모델의 설계도면
config = BertConfig.from_pretrained('bert-base-uncased')

# 원하는 파라미터를 수정할 수 있다.
config = BertConfig.from_pretrained('bert-base-uncased',
                                    num_hidden_layers=6,      # 12 -> 6개로 축소
                                    hidden_dropout_prob=0.2,   # Dropout 0.1 -> 0.2로 변경
                                    output_hidden_states=True
                                    )
# 파라미터를 수정하여 모델과 토크나이저에 적용할 수 있다.
bert = BertModel.from_pretrained("bert-base-uncased", config = config)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", config = config)
# 이때, 모델의 가중치가 바뀌도록 설정됐다면 사전학습을 다시 해야한다.


# 아예 새로운 모델을 선언할 수도 있다.
bert2 = BertModel(config)

# config 저장 및 로드
config.save_pretrained("./my_bert_config")
loaded_config = BertConfig.from_pretrained("./my_bert_config")

# 사전학습된 모델은 로드할 때 config가 암묵적으로 로드된다.
bert = BertModel.from_pretrained("bert-base-uncased")
bert.config

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                                                       | Status     |  | 
--------------------------------------------------------------------------+------------+--+-
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.attention.output.dense.weight     | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.attention.self.key.bias           | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.attention.output.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias                                      | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.attention.self.value.weight       | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.output.dense.bias                 | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.intermediate.dense.bias           | UNEXPECTED |  | 
bert.encoder.layer.{6, 7, 8, 9, 10, 11}.attention.self.value.bias         | UNEXPECTED |  | 
bert.enco

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.16.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [ ]:
a = copy.deepcopy(bert.pooler)

In [ ]:
# 모델 구조 살펴보기
# 선언된 모델은 직접 프린트하여 구조를 확인할 수 있다.
# inplace = False는 드롭아웃에서 새로운 메모리 주소를 할당하는 변수
# elementwise_affine=True는 Layer_norm에서 학습 가능한 파라미터를 곱하고 더하겠다는 뜻
# 멀티 헤드에 대한 표현은 명시적으로 되어있지 않지만, 어텐션 단계에서 이루어진다.

print(bert)

# 객체를 선언할 때 필요한 인자들은 inspect나 help()를 활용하여 볼 수 있다.
sig = inspect.signature(BertConfig)
print(sig)

sig2 = inspect.signature(BertModel)
print(sig2)

# 특정 레이어나 값만 떼서 확인할 수 있다.
print(bert.pooler.dense.out_features)
print(bert.embeddings)
print(bert.encoder.layer[3])

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
# 파라미터 조작
# 학습 가능한 파라미터는 다음과 같이 확인할 수 있다.
for name, param in bert.named_parameters():
  print(name)

# 파라미터 동결과 설정
'''
# 1. 모델 전체 파라미터 동결
for param in model.parameters():
    param.requires_grad = False

# 2. 상위 2개 레이어(10번, 11번)만 학습 해제 (Unfreeze)
for layer in model.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# 3. Pooler 레이어도 학습에 포함하고 싶은 경우
for param in model.pooler.parameters():
    param.requires_grad = True
'''

embeddings.word_embeddings.weight
embeddings.position_embeddings.weight
embeddings.token_type_embeddings.weight
embeddings.LayerNorm.weight
embeddings.LayerNorm.bias
encoder.layer.0.attention.self.query.weight
encoder.layer.0.attention.self.query.bias
encoder.layer.0.attention.self.key.weight
encoder.layer.0.attention.self.key.bias
encoder.layer.0.attention.self.value.weight
encoder.layer.0.attention.self.value.bias
encoder.layer.0.attention.output.dense.weight
encoder.layer.0.attention.output.dense.bias
encoder.layer.0.attention.output.LayerNorm.weight
encoder.layer.0.attention.output.LayerNorm.bias
encoder.layer.0.intermediate.dense.weight
encoder.layer.0.intermediate.dense.bias
encoder.layer.0.output.dense.weight
encoder.layer.0.output.dense.bias
encoder.layer.0.output.LayerNorm.weight
encoder.layer.0.output.LayerNorm.bias
encoder.layer.1.attention.self.query.weight
encoder.layer.1.attention.self.query.bias
encoder.layer.1.attention.self.key.weight
encoder.layer.1.attention.self.key

'\n# 1. 모델 전체 파라미터 동결\nfor param in model.parameters():\n    param.requires_grad = False\n\n# 2. 상위 2개 레이어(10번, 11번)만 학습 해제 (Unfreeze)\nfor layer in model.encoder.layer[-2:]:\n    for param in layer.parameters():\n        param.requires_grad = True\n\n# 3. Pooler 레이어도 학습에 포함하고 싶은 경우\nfor param in model.pooler.parameters():\n    param.requires_grad = True\n'

In [ ]:
tokenizer('hi', return_tensors = 'pt')

{'input_ids': tensor([[ 101, 7632,  102]]), 'token_type_ids': tensor([[0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1]])}

In [ ]:
# 선언한 모델로 추론하기
input = tokenizer('hi', return_tensors = 'pt')
bert(**input) # 딕셔너리로 형태로 선언된 자료를 키워드 인수로 풀어줌

# 언패킹의 기능
# *
# 1. 남아있는 데이터 몰아주기
n = [1,2,3,4,5]
a, *b, c = n
b

# 2. 함수 호출 시 데이터 풀어주기
def add_three(a,b,c):
  return a+b+c
n = [1,2,3]
add_three(*n)

# **
# 딕셔너리를 키워드 인수로 풀어버릴 떄 (key = value)
def profile(name, job, age):
  print(f'{name},{job},{age}')
prof = {
    'name' : '홍길동',
    'job' : '회사원',
    'age' : 20
}
profile(**prof)



홍길동,회사원,20


In [ ]:
# 모델 내의 차원이 어떻게 조작되는지 확인하기
bert.eval()
from torchinfo import summary
summary(bert, input_data=dict(input)) #또는 **input

Layer (type:depth-idx)                             Output Shape              Param #
BertModel                                          [1, 768]                  --
├─BertEmbeddings: 1-1                              [1, 3, 768]               --
│    └─Embedding: 2-1                              [1, 3, 768]               23,440,896
│    └─Embedding: 2-2                              [1, 3, 768]               1,536
│    └─Embedding: 2-3                              [1, 3, 768]               393,216
│    └─LayerNorm: 2-4                              [1, 3, 768]               1,536
│    └─Dropout: 2-5                                [1, 3, 768]               --
├─BertEncoder: 1-2                                 [1, 3, 768]               --
│    └─ModuleList: 2-6                             --                        --
│    │    └─BertLayer: 3-1                         [1, 3, 768]               7,087,872
│    │    └─BertLayer: 3-2                         [1, 3, 768]               7,087,872
│ 

In [ ]:
# 토크나이저의 종류
'''
자동 선언 방식 - AutoTokenizer : 범용적으로 사용되는 토크나이저 클래스. Fast 버전을 자동으로 우선하여 실행한다.
명시적 선언 방식 - BertTokenizer
커스텀 메서드를 직접 확장해야하는 방식이 아니라면 AutoTokenizer를 사용한다.
'''
# output_hidden_states=True
# output_attentions=True

bert = BertModel.from_pretrained('bert-base-uncased',
                                 output_attentions=True,
                                 output_hidden_states=True)
input = tokenizer('I am a student', return_tensors = 'pt')
outputs = bert(**input)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(tensor([[[[9.3869e-02, 7.9542e-02, 6.6293e-02, 2.3627e-01, 7.2882e-02,
            4.5114e-01],
           [3.1478e-01, 2.2110e-01, 1.2005e-01, 4.8374e-02, 1.6238e-01,
            1.3331e-01],
           [8.0831e-02, 5.4355e-01, 5.3321e-02, 1.4888e-02, 1.5940e-01,
            1.4802e-01],
           [1.9576e-01, 1.8203e-01, 1.6908e-01, 1.5565e-01, 1.7817e-01,
            1.1931e-01],
           [1.5427e-01, 3.2326e-01, 1.5094e-01, 9.4742e-02, 1.1441e-01,
            1.6237e-01],
           [1.9499e-01, 1.4526e-01, 5.5344e-02, 2.2274e-01, 9.8450e-02,
            2.8322e-01]],
 
          [[9.3965e-01, 2.1992e-02, 5.3800e-03, 1.6562e-02, 8.1502e-03,
            8.2670e-03],
           [1.5172e-02, 3.9312e-02, 2.0806e-01, 4.1500e-02, 6.2281e-01,
            7.3140e-02],
           [1.1028e-01, 8.3025e-02, 1.0685e-01, 1.7536e-01, 2.8953e-01,
            2.3496e-01],
           [4.1873e-02, 1.5679e-01, 2.0730e-01, 2.2822e-01, 2.9260e-01,
            7.3221e-02],
           [2.6166e-02, 4.7

In [ ]:
outputs.hidden_states[0].shape # 레이어는 튜플 # Bert Layer

torch.Size([1, 6, 768])

In [ ]:
outputs.attentions[0].shape # 레이어는 튜플 #(batch, head, length. length)

torch.Size([1, 12, 6, 6])

In [ ]:
# 다른 모델들
from transformers import BertForSequenceClassification, BertForTokenClassification, BertForQuestionAnswering

MODEL_NAME = 'bert-base-uncased'

model1 = BertModel.from_pretrained(MODEL_NAME)
model2 = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = 5)
model3 = BertForQuestionAnswering.from_pretrained(MODEL_NAME)

inspect.signature(BertForQuestionAnswering)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

<Signature (config)>

In [ ]:
# 데이터 콜레이터 선정 방법
'''
DataCollatorForLanguageModeling : 라벨 생성
DataCollatorForCompletionOnlyLM : Chat Template
DataCollatorWithPadding
DataCollatorForSeq2Seq

import torch
from transformers import AutoTokenizer
from trl import DataCollatorForCompletionOnlyLM

messages = [
    {"role": "user", "content": "한국의 수도는 어디인가요?"},
    {"role": "assistant", "content": "한국의 수도는 서울입니다."}
]
# Qwen, LLaMA-3 등의 Chat Template 적용 시
# Assistant 응답 바로 직전에 "<|im_start|>assistant\n" 같은 특수 구문이 생성됨
formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)

# 3. Assistant 응답 시작 템플릿 지정
# 사용하는 모델의 Chat Template 결과에 맞춰 response_template을 설정해야 합니다.
# 예: Qwen 계열 -> "<|im_start|>assistant\n"
# 예: LLaMA-3 계열 -> "<|start_header_id|>assistant<|end_header_id|>\n\n"
response_template = "<|im_start|>assistant\n"

# 4. CompletionOnly Data Collator 생성
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer
)

'''

In [ ]:
# 콜레이터의 기능
'''
1. 패딩
2. 디코더 모델의 경우 labels 적용
3. 인코더 모델의 경우 token_type_ids 적용
4. 어텐션 마스크 생성
5. 텐서 변환
'''


# 모델, 토크나이저, config 선정 방법
'''
config : AutoConfig, BertConfig, LlamaConfig, GPT2Config, MIstralConfig 등..
model: AutoModel BertModel, GPP2Model, GPT2LMHeadModel, LlamaForCausalLM 등..
tokenizer: AutoTokenizer, GPT2Tokenizer, LlamaTokenizer, BertTokenizer


Auto를 쓰면 안 되는 경우
1. 모델을 새로 정의할 때
2. 일부 신규 모델에서 trust_remote_code = True를 써야하는데, 기업 보안 수칙 상 금지될 때
3. 멀티 모달 (AutoProcessor)
4. 동적할당이 되지 않는 상황일 때
5. 특정 라이브러리 전용 메서드를 써야할 때
'''

In [ ]:
# Generate, pipeline
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-2B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-2B")



config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [ ]:
generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    torch_dtype = torch.bfloat16,
    device_map = 'auto'
)

output = generator(
    "Hi",
    max_new_tokens = 150,
    temperature = 0.7,
    do_sample = True
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
output = model.generate(
        **tokenizer("hi", return_tensors = 'pt'),
        max_new_tokens=10,
        do_sample=False

    )
print(tokenizer.decode(output))

['hi, I am a 20 year old male']


In [ ]:
print(output)

[{'generated_text': "Hi, I'm working on a project where I need to parse a log file using regular expressions. I'm having trouble with the regex I've written. Can you help me debug and fix the issue?\n\nHere's the issue: The regex is not matching because the string is being passed as a string in the regex, which is expected to be a literal. However, the `String` object itself is not being used. The issue is likely that the input string is being passed as a string literal, which is expected to be a literal. However, the `String` object itself is not being used. The issue is likely that the input string is being passed as a string literal, which is expected to be a literal. However, the `"}]
